# Compact 3D CNN: first Colab GPU pilot

Run from the top, one section at a time. The previous preparation has passed.
This notebook fits an engineering baseline, not a clinically validated system.
RD is the positive class (1); normal/PVD are non-RD (0).

**Before section 1:** save your previous notebook, select **Runtime → Change runtime
type → GPU** in browser Colab (T4 if available), and reconnect. In VS Code select
the corresponding Colab GPU kernel. Switching runtime clears Python state; run
this notebook from the top. Free GPUs and session duration are not guaranteed.

The notebook is standalone: no upstream clone, local project installation, or
previous notebook variables are needed. Colab's managed Python and installed CUDA
PyTorch are used deliberately; local `uv` environments do not configure Colab.
If an import fails, stop and report it instead of replacing the CUDA installation.

Order: GPU check → fixed manifests → four training clips → forward/backward and
overfit checks → remaining pilot/validation clips → five epochs → save results.
The larger download is gated on the overfit check. No test CSV is opened and no
test videos or metrics are used. The supplied clip-level split can have patients
in more than one partition; these results cannot measure unseen-patient performance.

References: [baseline plan](../docs/BASELINE_PLAN.md),
[Colab FAQ](https://research.google.com/colaboratory/faq.html),
[PyTorch reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html),
[ERDES paper](https://arxiv.org/html/2508.04735v2).


## 1. Confirm the GPU and storage

Expected: `CUDA available: True` and a GPU name. A CPU runtime stops here.
Results are written after each epoch to a new run directory. `/content` is
temporary. For persistent checkpoints, set `USE_GOOGLE_DRIVE = True` before
running this cell and authorize the standard Colab Drive mount. Videos and
preprocessed arrays always stay on runtime storage. Otherwise download the
results ZIP at the end before disconnecting. A notebook save alone does not
preserve its checkpoint files. Interrupted runs remain partial; rerunning from
the top creates a fresh run, without automatic resume or overwriting a prior run.


In [1]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import hashlib
import io
import json
import random
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from urllib.parse import quote
from urllib.request import urlopen

import cv2
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import average_precision_score, roc_auc_score
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import huggingface_hub
from huggingface_hub import HfApi
from IPython.display import display

print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime, reconnect, then run from section 1.')
device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))

USE_GOOGLE_DRIVE = False
cwd = Path.cwd()
project_root = next((p for p in (cwd, *cwd.parents)
    if (p / 'pyproject.toml').is_file() and (p / 'notebooks').is_dir()), None)
if project_root is None:
    if not Path('/content').is_dir():
        raise RuntimeError('Open from the project folder or use a Colab runtime.')
    project_root = Path('/content')
data_dir = project_root / 'data' / 'erdes'
results_root = project_root / 'outputs' / 'baseline_training'
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    results_root = Path('/content/drive/MyDrive/rd-ultrasound-ml/outputs/baseline_training')
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
output_dir = results_root / run_id
output_dir.mkdir(parents=True, exist_ok=False)
data_dir.mkdir(parents=True, exist_ok=True)
runtime = {'python': sys.version, 'numpy': np.__version__, 'pandas': pd.__version__,
    'opencv': cv2.__version__, 'sklearn': sklearn.__version__, 'torch': torch.__version__,
    'huggingface_hub': huggingface_hub.__version__, 'cuda': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version(), 'gpu': torch.cuda.get_device_name(0)}
print(runtime)
print('Results:', output_dir)
print('Videos/arrays:', data_dir)

def sha256(content):
    return hashlib.sha256(content).hexdigest()

def save_json(name, value):
    path = output_dir / name
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + '\n', encoding='utf-8')
    temporary.replace(path)



CUDA available: True
GPU: Tesla T4
{'python': '3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]', 'numpy': '2.1.3', 'pandas': '2.2.3', 'opencv': '5.0.0', 'sklearn': '1.6.1', 'torch': '2.11.0+cu130', 'huggingface_hub': '1.33.0', 'cuda': '13.0', 'cudnn': 92700, 'gpu': 'Tesla T4'}
Results: /content/outputs/baseline_training/20261028T193629481701Z
Videos/arrays: /content/data/erdes


### Download helper — safe to rerun after an HTTP error

This cell only defines the downloader. It does not reset your run, manifests,
GPU state, or diagnostic result. If video acquisition stops with a temporary
HTTP error, run **this helper cell**, then rerun the failed acquisition cell.
Keep the same Colab runtime; do not rerun section 1's runtime/storage cell.
Already downloaded clips are hash-checked and reused, so progress may start
counting from 1 again without downloading those clips again.

Temporary HTTP 408/429/500/502/503/504, connection errors, and incomplete reads
get at most five attempts with increasing waits. Server Retry-After delays
are respected; requests to wait over 60 seconds stop with instructions.
Permanent HTTP errors and checksum mismatches stop immediately. No clip is
skipped or substituted. A persistent failure names the affected clip.


In [2]:
import hashlib
import time
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from http.client import IncompleteRead
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

def download_checked(url, path, expected_hash, algorithm='sha256', expected_bytes=None, max_attempts=5):
    path = Path(path)
    if max_attempts < 1:
        raise ValueError('max_attempts must be positive')
    if path.exists():
        content = path.read_bytes()
    else:
        for attempt in range(1, max_attempts + 1):
            retry_after = None
            try:
                with urlopen(url, timeout=90) as response:
                    content = response.read()
                break
            except HTTPError as error:
                status = error.code
                retry_after = error.headers.get('Retry-After') if error.headers else None
                error.close()
                if status not in {408, 429, 500, 502, 503, 504}:
                    raise
                failure = f'HTTP {status}'
            except (URLError, TimeoutError, ConnectionError, IncompleteRead) as error:
                failure = type(error).__name__
            if attempt == max_attempts:
                raise RuntimeError(
                    f'Download failed for {path.name}: {failure} after {max_attempts} attempts. '
                    'Rerun the acquisition cell in this runtime to reuse verified cached clips.'
                ) from None
            delay = min(60.0, float(2 ** attempt))
            if retry_after:
                try:
                    server_delay = float(retry_after)
                except ValueError:
                    try:
                        server_delay = (parsedate_to_datetime(retry_after) - datetime.now(timezone.utc)).total_seconds()
                    except (TypeError, ValueError, OverflowError):
                        server_delay = 0.0
                if server_delay > 60:
                    raise RuntimeError(
                        f'{path.name}: {failure}; server requests a wait of {server_delay:.0f}s. '
                        'Wait that long, then rerun the acquisition cell; cached clips are preserved.'
                    ) from None
                delay = max(delay, server_delay)
            print(f'{path.name}: {failure}; retry {attempt + 1}/{max_attempts} in {delay:g}s', flush=True)
            time.sleep(delay)
    if hashlib.new(algorithm, content).hexdigest() != expected_hash:
        raise ValueError(f'Checksum mismatch: {path.name}; cached file was not overwritten')
    if expected_bytes is not None and len(content) != expected_bytes:
        raise ValueError(f'Unexpected size: {path.name}')
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + '.part')
        temporary.write_bytes(content)
        temporary.replace(path)
    return content

print('Download helper ready: retry temporary HTTP failures; reuse checksum-verified cached clips.')


Download helper ready: retry temporary HTTP failures; reuse checksum-verified cached clips.


## 2. Freeze the run configuration and seeds

The embedded configuration matches `configs/baseline_001_training.json`.
Keep the initial experiment unchanged: 16 grayscale frames, 128×128, no
augmentation, unweighted BCE loss, Adam 0.001, batch size 8, five epochs.
The four-clip diagnostic uses its separately recorded settings and its weights
are discarded. Strict determinism raises an error for unsupported operations;
report that error rather than silently disabling the setting.


In [3]:
config = {'experiment_id': 'baseline_001_pilot',
 'status': 'implemented_not_run_on_gpu',
 'task': 'non_rd_vs_rd',
 'label_mapping': {'non_rd': 0, 'rd': 1},
 'seed': 42,
 'split_policy': 'preserve official train/val/test membership; no test use during '
                 'development',
 'preprocessing': {'num_frames': 16,
                   'height': 128,
                   'width': 128,
                   'frame_sampling': 'uniform frame indices using rint(linspace(0, '
                                     'N-1, T)); repeats for short clips',
                   'grayscale': 'arithmetic channel mean',
                   'padding': 'symmetric zero padding to square',
                   'resize': 'OpenCV INTER_AREA',
                   'normalization': 'divide by 255',
                   'layout': 'B,C,T,H,W',
                   'dtype': 'float32',
                   'augmentation': 'none'},
 'model': {'name': 'compact_3d_cnn_from_scratch',
           'channels': [8, 16, 32],
           'head': 'global average pooling and one logit',
           'pretrained': False,
           'blocks': 'Conv3d(kernel=3,stride=2,padding=1,bias=True), ReLU; repeated '
                     'for each channel count',
           'pool': 'global mean over temporal and spatial dimensions',
           'output': 'Linear(32,1), no sigmoid'},
 'training': {'device': 'Colab CUDA GPU',
              'optimizer': 'Adam',
              'learning_rate': 0.001,
              'batch_size': 8,
              'epochs': 5,
              'loss': 'unweighted BCEWithLogitsLoss',
              'pilot_training_target': 256,
              'validation': 'all 431 supplied validation clips',
              'pilot_selection': 'deterministic proportional class sample within '
                                 'official training only; record manifest before '
                                 'fitting',
              'pilot_class_counts': {'0': 232, '1': 24},
              'num_workers': 0,
              'shuffle': True,
              'precision': 'float32',
              'deterministic_algorithms': True},
 'evaluation': {'positive_class': 'rd',
                'threshold': 0.5,
                'checkpoint_selection': 'highest validation average precision; '
                                        'earliest epoch on ties',
                'metrics': ['TP',
                            'TN',
                            'FP',
                            'FN',
                            'sensitivity',
                            'specificity',
                            'precision',
                            'F1',
                            'balanced_accuracy',
                            'average_precision',
                            'ROC_AUC',
                            'accuracy'],
                'final_test': 'reserved; no test loading, inference, tuning or '
                              'checkpoint selection'},
 'limitations': ['clip-level splits allow same-patient overlap; no unseen-patient '
                 'generalization claim',
                 'eight inspection clips are not a training dataset or a metric '
                 'estimate',
                 'pilot is not a reproduction of the paper or a final experiment'],
 'preparation_config_sha256': '40753e7e9a4ba8bca86b445307a26c85e2087ac3740b93cf1525ae4d2fe7a1e1',
 'diagnostics': {'clips_per_class': 2,
                 'max_steps': 300,
                 'learning_rate': 0.003,
                 'loss_ratio_required': 0.5,
                 'training_accuracy_required': 1.0,
                 'selection': 'first two per class in the ranked pilot manifest; '
                              'training only'}}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True)

seed_everything(config['seed'])
runtime['determinism'] = {'algorithms': torch.are_deterministic_algorithms_enabled(),
    'cudnn_benchmark': torch.backends.cudnn.benchmark,
    'cudnn_deterministic': torch.backends.cudnn.deterministic,
    'cublas_workspace_config': os.environ['CUBLAS_WORKSPACE_CONFIG'],
    'matmul_tf32': torch.backends.cuda.matmul.allow_tf32,
    'cudnn_tf32': torch.backends.cudnn.allow_tf32}
save_json('config.json', config)
config_hash = sha256((output_dir / 'config.json').read_bytes())
save_json('runtime.json', runtime)
save_json('run_status.json', {'status': 'configured', 'test_used': False})
print('Configuration SHA256:', config_hash)


Configuration SHA256: a2417d501c1f906ce2804def0f9af0cbc910f8fe837c02a2c82ae66d1006615f


## 3. Record the official training subset and validation manifest

Only `train.csv` and `val.csv` are read from the checksum-verified release
archive. Rank training clip IDs by SHA256 of `42:<clip_id>` within each binary
class; retain 232 non-RD and 24 RD. This is a development subset of the official
training split, not a new train/validation split. Use all 431 validation rows.
Record manifests and source hashes before any weights are fitted.


In [4]:
def join_partition(content, metadata, expected_counts):
    frame = pd.read_csv(io.BytesIO(content))
    if not frame['path'].is_unique or frame['label'].value_counts().to_dict() != expected_counts:
        raise ValueError('Unexpected official split counts or duplicate paths')
    joined = frame.merge(metadata, left_on='path', right_on='file_path', how='left', validate='one_to_one')
    if joined['clip_id'].isna().any():
        raise ValueError('Missing metadata')
    if not joined['label'].map({0: 'non_rd', 1: 'rd'}).eq(joined['diagnostic_class']).all():
        raise ValueError('Label/metadata disagreement')
    for path in joined['file_path']:
        relative = PurePosixPath(path)
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError('Unsafe relative video path')
    return joined

def select_pilot(training, counts, seed):
    selected = []
    for label, count in sorted(counts.items()):
        candidates = training.loc[training['label'].eq(int(label))].copy()
        candidates['selection_rank'] = candidates['clip_id'].map(
            lambda clip_id: sha256(f'{seed}:{clip_id}'.encode()))
        if len(candidates) < count:
            raise ValueError('Insufficient training clips for requested subset')
        selected.append(candidates.sort_values(['selection_rank', 'clip_id']).head(count))
    return pd.concat(selected, ignore_index=True)


In [5]:
release_url = 'https://zenodo.org/records/18644370'
metadata_bytes = download_checked(f'{release_url}/files/erdes_metadata.csv?download=1',
    data_dir / 'erdes_metadata.csv', '95828fb5bdffdbf8173b5bcfc99e8148', algorithm='md5')
archive_bytes = download_checked(f'{release_url}/files/splits.zip?download=1',
    data_dir / 'splits.zip', 'c5630e3f056ce358dcd384aaed946e64', algorithm='md5')
metadata = pd.read_csv(io.BytesIO(metadata_bytes))
assert len(metadata) == 5381 and metadata['clip_id'].is_unique and metadata['file_path'].is_unique
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    train_bytes = archive.read('splits/non_rd_vs_rd/train.csv')
    val_bytes = archive.read('splits/non_rd_vs_rd/val.csv')
training = join_partition(train_bytes, metadata, {0: 3511, 1: 362})
validation = join_partition(val_bytes, metadata, {0: 391, 1: 40})
assert not set(training['path']) & set(validation['path'])
pilot = select_pilot(training, config['training']['pilot_class_counts'], config['seed'])
assert len(pilot) == 256 and pilot['clip_id'].is_unique
assert set(pilot['path']).issubset(set(training['path']))
tiny = pilot.groupby('label', sort=True).head(config['diagnostics']['clips_per_class']).copy()
manifest_hashes = {}
for name, frame in [('train_manifest.csv', pilot), ('validation_manifest.csv', validation),
                    ('diagnostic_manifest.csv', tiny)]:
    frame.to_csv(output_dir / name, index=False, lineterminator='\n')
    manifest_hashes[name] = sha256((output_dir / name).read_bytes())
(output_dir / 'official_train.csv').write_bytes(train_bytes)
(output_dir / 'official_val.csv').write_bytes(val_bytes)
save_json('manifests.json', {'sha256': manifest_hashes,
    'metadata_sha256': sha256(metadata_bytes), 'split_archive_sha256': sha256(archive_bytes),
    'official_train_sha256': sha256(train_bytes), 'official_val_sha256': sha256(val_bytes),
    'train_count': len(pilot), 'validation_count': len(validation), 'test_used': False})
display(pilot.groupby('label').size().rename('pilot clips'))
display(validation.groupby('label').size().rename('validation clips'))
print('Manifests saved. No test CSV opened.')


,pilot clips
label,
0,232
1,24


,validation clips
label,
0,391
1,40


Manifests saved. No test CSV opened.


## 4. Preprocessing, model, and metrics

Preprocessing is identical to the completed preparation notebook. All source
frames are decoded to verify the frame count; selected positions are padded,
resized, and scaled. There are no learned preprocessing statistics.

The model has three strided Conv3d/ReLU blocks (8, 16, 32 channels), a global
mean, and a single raw logit. Using a direct mean avoids AdaptiveAvgPool3d's
unsupported deterministic CUDA backward operation. Sigmoid is applied only for probabilities.
Metrics use RD=1 and threshold 0.5. Average precision uses scikit-learn's
recall-weighted definition, not trapezoidal PR-AUC. Undefined metrics are JSON
`null` with reasons. An always-non-RD classifier is reported on the same rows.

**Recovery from a pixel-range error:** rerun the preprocessing function cell below,
then rerun the updated section 7 cell in the same runtime. It reuses downloaded
videos and rebuilds incomplete arrays in the existing run folder. The strict
clip range check remains active; only float32 resize drift up to `1e-6` is
corrected. Larger errors report the affected clip and frame-value range.


In [6]:
def uniform_frame_indices(frame_count, num_frames):
    if frame_count <= 0 or num_frames <= 0:
        raise ValueError("frame_count and num_frames must be positive")
    return np.rint(np.linspace(0, frame_count - 1, num_frames)).astype(np.int64)

def prepare_frame(frame, image_size):
    if image_size <= 0:
        raise ValueError("image_size must be positive")
    if frame.ndim != 3 or frame.shape[2] != 3:
        raise ValueError("Expected an OpenCV H,W,3 frame")
    if frame.dtype != np.uint8:
        raise ValueError("Expected uint8 decoded pixels in [0,255]")
    gray = frame.mean(axis=2, dtype=np.float32)
    height, width = gray.shape
    side = max(height, width)
    top = (side - height) // 2
    left = (side - width) // 2
    padded = np.pad(gray, ((top, side - height - top), (left, side - width - left)))
    resized = cv2.resize(padded, (image_size, image_size), interpolation=cv2.INTER_AREA)
    scaled = (resized / np.float32(255)).astype(np.float32)
    # INTER_AREA float32 weights can move a saturated pixel a few ULPs beyond 1.
    # Correct only numerical drift, while rejecting nonfinite or materially bad values.
    tolerance = 1e-6
    minimum, maximum = float(scaled.min()), float(scaled.max())
    if not np.isfinite(scaled).all() or minimum < -tolerance or maximum > 1 + tolerance:
        raise ValueError(f"Resized frame outside finite [0,1] bounds: min={minimum}, max={maximum}")
    return np.clip(scaled, np.float32(0), np.float32(1), out=scaled)

def prepare_clip(path, expected_frames, num_frames=16, image_size=128):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    indices = uniform_frame_indices(expected_frames, num_frames)
    targets = set(indices.tolist())
    selected = {}
    capture = cv2.VideoCapture(str(path))
    count = 0
    try:
        if not capture.isOpened():
            raise ValueError(f"Cannot open video: {path.name}")
        while True:
            ok, frame = capture.read()
            if not ok:
                break
            if count in targets:
                selected[count] = prepare_frame(frame, image_size)
            count += 1
    finally:
        capture.release()
    if count != expected_frames:
        raise ValueError(f"{path.name}: decoded {count} frames; metadata expects {expected_frames}")
    clip = np.stack([selected[int(i)] for i in indices], axis=0)[None, ...]
    if not np.isfinite(clip).all() or clip.min() < 0 or clip.max() > 1:
        raise ValueError(f"{path.name}: preprocessed values must be finite and in [0,1]; "
                         f"min={float(clip.min())}, max={float(clip.max())}")
    return clip, indices


In [7]:
class Compact3DCNN(nn.Module):
    def __init__(self, channels=(8, 16, 32)):
        super().__init__()
        blocks = []
        incoming = 1
        for outgoing in channels:
            blocks.extend([nn.Conv3d(incoming, outgoing, 3, stride=2, padding=1), nn.ReLU()])
            incoming = outgoing
        self.features = nn.Sequential(*blocks)
        self.head = nn.Linear(incoming, 1)

    def forward(self, x):
        return self.head(self.features(x).mean(dim=(2, 3, 4))).squeeze(1)

def classification_metrics(labels, probabilities, threshold=0.5):
    y = np.asarray(labels)
    p = np.asarray(probabilities, dtype=np.float64)
    if y.ndim != 1 or p.shape != y.shape or len(y) == 0 or not np.isin(y, [0, 1]).all():
        raise ValueError('Expected non-empty matching binary label/probability vectors')
    if not np.isfinite(p).all() or np.any((p < 0) | (p > 1)):
        raise ValueError('Probabilities must be finite and in [0,1]')
    predicted = p >= threshold
    tp = int(np.sum((y == 1) & predicted))
    tn = int(np.sum((y == 0) & ~predicted))
    fp = int(np.sum((y == 0) & predicted))
    fn = int(np.sum((y == 1) & ~predicted))
    reasons = {}
    def ratio(name, numerator, denominator):
        if denominator == 0:
            reasons[name] = 'zero denominator'
            return None
        return float(numerator / denominator)
    sensitivity = ratio('sensitivity', tp, tp + fn)
    specificity = ratio('specificity', tn, tn + fp)
    result = {'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
        'sensitivity': sensitivity, 'specificity': specificity,
        'precision': ratio('precision', tp, tp + fp), 'F1': ratio('F1', 2 * tp, 2 * tp + fp + fn),
        'accuracy': float((tp + tn) / len(y)), 'threshold': threshold, 'positive_class': 'rd'}
    result['balanced_accuracy'] = None if sensitivity is None or specificity is None else (sensitivity + specificity) / 2
    if result['balanced_accuracy'] is None:
        reasons['balanced_accuracy'] = 'requires both classes'
    if set(y.tolist()) == {0, 1}:
        result['average_precision'] = float(average_precision_score(y, p))
        result['ROC_AUC'] = float(roc_auc_score(y, p))
    else:
        result.update(average_precision=None, ROC_AUC=None)
        reasons.update(average_precision='requires both classes for this protocol', ROC_AUC='requires both classes')
    result['undefined_reasons'] = reasons
    return result

def improves_average_precision(score, best_score):
    return np.isfinite(score) and score > best_score

def make_model():
    return Compact3DCNN(config['model']['channels']).to(device)

criterion = nn.BCEWithLogitsLoss()
model = make_model()
parameter_count = sum(parameter.numel() for parameter in model.parameters())
save_json('model.json', {'architecture': str(model), 'parameter_count': parameter_count})
print(model)
print('Trainable parameters:', parameter_count)
del model


Compact3DCNN(
  (features): Sequential(
    (0): Conv3d(1, 8, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
    (1): ReLU()
    (2): Conv3d(8, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
    (3): ReLU()
    (4): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
    (5): ReLU()
  )
  (head): Linear(in_features=32, out_features=1, bias=True)
)
Trainable parameters: 17585


## 5. Download only four training clips for the diagnostic

Source sizes and SHA256 checksums are obtained for the selected paths at the
same immutable author-dataset revision used in inspection. Cached and downloaded
bytes must match. Corrupt caches stop the run without being overwritten.
No whole-dataset archive or authentication token is needed for this public source.
The source manifest retains the author card's CC-BY-4.0 declaration and citation.


In [8]:
hf_revision = '7867e2504ed3845a46ff7790a2f8c3ffcdc08947'
hf_dataset = 'https://huggingface.co/datasets/pcvlab/erdes'
api = HfApi(token=False)
sources = {}

def record_sources(frame):
    missing = [f'clips/{path}' for path in frame['file_path'] if path not in sources]
    for start in range(0, len(missing), 50):
        requested = missing[start:start + 50]
        entries = api.get_paths_info('pcvlab/erdes', paths=requested, repo_type='dataset',
                                    revision=hf_revision, token=False)
        by_path = {entry.path: entry for entry in entries}
        if set(by_path) != set(requested):
            raise ValueError('Source API did not return every selected path')
        for path in requested:
            entry = by_path[path]
            if entry.lfs is None or len(entry.lfs.sha256) != 64 or entry.size != entry.lfs.size:
                raise ValueError(f'Missing source SHA256/size: {path}')
            sources[path.removeprefix('clips/')] = {'sha256': entry.lfs.sha256, 'bytes': int(entry.size),
                'url': f'{hf_dataset}/resolve/{hf_revision}/{quote(path, safe="/")}'}
    save_json('video_sources.json', {'dataset': hf_dataset, 'revision': hf_revision,
        'license_declaration': 'cc-by-4.0 (author dataset card)',
        'paper': 'https://arxiv.org/html/2508.04735v2', 'files': sources})

def acquire_videos(frame):
    print('Selected source bytes (including cached):', sum(sources[path]['bytes'] for path in frame['file_path']))
    for index, row in enumerate(frame.itertuples(index=False), 1):
        source = sources[row.file_path]
        download_checked(source['url'], data_dir / 'clips' / row.file_path,
                         source['sha256'], expected_bytes=source['bytes'])
        if index == 1 or index % 25 == 0 or index == len(frame):
            print(f'Hash-verified {index}/{len(frame)} clips', flush=True)

record_sources(tiny)
acquire_videos(tiny)
params = config['preprocessing']
tiny_x = torch.from_numpy(np.stack([prepare_clip(data_dir / 'clips' / row.file_path,
    int(row.frame_count), params['num_frames'], params['height'])[0]
    for row in tiny.itertuples(index=False)])).to(device)
tiny_y = torch.tensor(tiny['label'].to_numpy(), dtype=torch.float32, device=device)
assert tuple(tiny_x.shape) == (4, 1, 16, 128, 128)
assert set(tiny_y.cpu().tolist()) == {0.0, 1.0}
print('Diagnostic tensor:', tuple(tiny_x.shape), 'range:', tiny_x.min().item(), tiny_x.max().item())


Selected source bytes (including cached): 1899479
Hash-verified 1/4 clips
Hash-verified 4/4 clips
Diagnostic tensor: (4, 1, 16, 128, 128) range: 0.0 0.9753789901733398


## 6. Forward/backward and four-clip overfit check — stop if it fails

Check finite logits, loss, and nonzero finite gradients. A fresh diagnostic
model then repeatedly sees these four training clips for at most 300 steps.
Pass requires loss at most half its initial value **and** all four training
labels predicted correctly at threshold 0.5. These numbers diagnose the code;
they are not validation performance. Failure is recorded and blocks the larger
download. Keep the failure evidence and investigate before changing settings.


In [9]:
diagnostics_passed = False
seed_everything(config['seed'])
smoke_model = make_model()
logits = smoke_model(tiny_x)
assert logits.shape == tiny_y.shape and torch.isfinite(logits).all()
loss = criterion(logits, tiny_y)
assert torch.isfinite(loss)
loss.backward()
gradients = [p.grad for p in smoke_model.parameters() if p.requires_grad]
assert all(g is not None and torch.isfinite(g).all() for g in gradients)
assert sum(g.abs().sum().item() for g in gradients) > 0
smoke_report = {'loss': loss.item(), 'logit_shape': list(logits.shape), 'finite_nonzero_gradients': True}
del smoke_model, logits, loss, gradients

seed_everything(config['seed'])
diagnostic_model = make_model()
settings = config['diagnostics']
diagnostic_optimizer = torch.optim.Adam(diagnostic_model.parameters(), lr=settings['learning_rate'])
with torch.no_grad():
    initial_loss = criterion(diagnostic_model(tiny_x), tiny_y).item()
diagnostic_history = []
diagnostic_start = time.perf_counter()
for step in range(1, settings['max_steps'] + 1):
    diagnostic_model.train()
    diagnostic_optimizer.zero_grad(set_to_none=True)
    loss = criterion(diagnostic_model(tiny_x), tiny_y)
    if not torch.isfinite(loss):
        raise ValueError('Nonfinite diagnostic loss')
    loss.backward()
    diagnostic_optimizer.step()
    diagnostic_model.eval()
    with torch.no_grad():
        logits = diagnostic_model(tiny_x)
        current_loss = criterion(logits, tiny_y).item()
        accuracy = ((logits >= 0) == tiny_y.bool()).float().mean().item()
    diagnostics_passed = (current_loss <= initial_loss * settings['loss_ratio_required']
                          and accuracy >= settings['training_accuracy_required'])
    row = {'step': step, 'loss': current_loss, 'accuracy': accuracy}
    diagnostic_history.append(row)
    if step == 1 or step % 20 == 0 or diagnostics_passed:
        print(row, flush=True)
        save_json('diagnostics.json', {'passed': diagnostics_passed, 'smoke': smoke_report,
            'initial_loss': initial_loss, 'history': diagnostic_history,
            'seconds': time.perf_counter() - diagnostic_start, 'test_used': False})
    if diagnostics_passed:
        break
save_json('diagnostics.json', {'passed': diagnostics_passed, 'smoke': smoke_report,
    'initial_loss': initial_loss, 'history': diagnostic_history,
    'seconds': time.perf_counter() - diagnostic_start, 'test_used': False})
save_json('run_status.json', {'status': 'diagnostics_passed' if diagnostics_passed else 'diagnostics_failed', 'test_used': False})
del diagnostic_model, diagnostic_optimizer, logits, loss, tiny_x, tiny_y
torch.cuda.empty_cache()
if not diagnostics_passed:
    raise RuntimeError('Four-clip overfit check failed. Stop and review diagnostics.json; do not run section 7.')
print('PASS: diagnostic weights discarded. The pilot will start from a fresh seeded model.')


{'step': 1, 'loss': 0.6938667893409729, 'accuracy': 0.5}
{'step': 20, 'loss': 0.5020955204963684, 'accuracy': 1.0}
{'step': 25, 'loss': 0.32425886392593384, 'accuracy': 1.0}
PASS: diagnostic weights discarded. The pilot will start from a fresh seeded model.


## 7. Download and preprocess the pilot and validation videos

This stage proceeds only after the diagnostic passes. Review the printed source
download size. It acquires 256 training and 431 validation clips, checking every
SHA256. Progress is printed every 25 clips. Stop on decode/count failures;
do not silently drop examples. Preprocessed float32 arrays use roughly 0.72 GB
on runtime disk through memory mapping. Each batch is copied into CPU memory,
then sent to the GPU; raw videos are decoded only once per run.


In [10]:
if not diagnostics_passed:
    raise RuntimeError('Complete the diagnostic successfully first.')
if (output_dir / 'history.json').exists() or (output_dir / 'last.pt').exists():
    raise RuntimeError('Training artifacts already exist. Start a fresh run before rebuilding its input arrays.')
save_json('preprocessing_implementation.json', {
    'version': 'float32_resize_bounds_v2', 'roundoff_tolerance': 1e-6,
    'correction': 'reject nonfinite/material out-of-range values, then clip numerical drift to [0,1]',
    'rerun_policy': 'rebuild complete arrays from the unchanged manifests before training',
})
all_selected = pd.concat([pilot, validation], ignore_index=True)
record_sources(all_selected)
acquire_videos(all_selected)
array_dir = data_dir / 'prepared_pilots' / run_id
array_dir.mkdir(parents=True, exist_ok=True)

def preprocess_partition(frame, name):
    shape = (len(frame), 1, params['num_frames'], params['height'], params['width'])
    path = array_dir / f'{name}.npy'
    array = np.lib.format.open_memmap(path, mode='w+', dtype=np.float32, shape=shape)
    checks = []
    for index, row in enumerate(frame.itertuples(index=False)):
        try:
            clip, indices = prepare_clip(data_dir / 'clips' / row.file_path, int(row.frame_count),
                                         params['num_frames'], params['height'])
        except ValueError as error:
            raise ValueError(f'{name} clip {index + 1}/{len(frame)} ({row.clip_id}): {error}') from error
        if clip.shape != shape[1:]:
            raise ValueError(f'Unexpected tensor shape for {row.clip_id}')
        array[index] = clip
        checks.append({'clip_id': row.clip_id, 'shape': list(clip.shape),
            'min': float(clip.min()), 'max': float(clip.max()), 'selected_indices': indices.tolist()})
        if (index + 1) % 25 == 0 or index + 1 == len(frame):
            print(f'Preprocessed {name}: {index + 1}/{len(frame)}', flush=True)
    array.flush()
    save_json(f'{name}_preprocessing.json', checks)
    return array

preprocessing_start = time.perf_counter()
train_array = preprocess_partition(pilot, 'train')
val_array = preprocess_partition(validation, 'validation')
preprocessing_seconds = time.perf_counter() - preprocessing_start

class PreparedClips(Dataset):
    def __init__(self, array, labels):
        self.array = array
        self.labels = np.asarray(labels, dtype=np.float32)
        if len(self.array) != len(self.labels):
            raise ValueError('Tensor/label length mismatch')

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return torch.from_numpy(np.array(self.array[index], copy=True)), torch.tensor(self.labels[index])

def make_loaders():
    generator = torch.Generator().manual_seed(config['seed'])
    batch_size = config['training']['batch_size']
    train_loader = DataLoader(PreparedClips(train_array, pilot['label']), batch_size=batch_size,
        shuffle=True, generator=generator, num_workers=0)
    val_loader = DataLoader(PreparedClips(val_array, validation['label']), batch_size=batch_size,
        shuffle=False, num_workers=0)
    return train_loader, val_loader

print('Prepared:', train_array.shape, val_array.shape)
print('Preprocessing seconds:', round(preprocessing_seconds, 1))


Selected source bytes (including cached): 248833155
Hash-verified 1/687 clips
Hash-verified 25/687 clips
Hash-verified 50/687 clips
Hash-verified 75/687 clips
Hash-verified 100/687 clips
Hash-verified 125/687 clips
Hash-verified 150/687 clips
Hash-verified 175/687 clips
Hash-verified 200/687 clips
Hash-verified 225/687 clips
Hash-verified 250/687 clips
Hash-verified 275/687 clips
Hash-verified 300/687 clips
Hash-verified 325/687 clips
Hash-verified 350/687 clips
Hash-verified 375/687 clips
Hash-verified 400/687 clips
Hash-verified 425/687 clips
Hash-verified 450/687 clips
Hash-verified 475/687 clips
Hash-verified 500/687 clips
Hash-verified 525/687 clips
Hash-verified 550/687 clips
Hash-verified 575/687 clips
Hash-verified 600/687 clips
Hash-verified 625/687 clips
Hash-verified 650/687 clips
Hash-verified 675/687 clips
Hash-verified 687/687 clips
Preprocessed train: 25/256
Preprocessed train: 50/256
Preprocessed train: 75/256
Preprocessed train: 100/256
Preprocessed train: 125/256
Prep

In [11]:
from pathlib import Path
import zipfile

root = Path("/content/outputs/baseline_training")
runs = [
    p for p in root.glob("*")
    if p.is_dir()
    and (p / "summary.json").is_file()
    and (p / "best.pt").is_file()
]

if not runs:
    print("No completed training run found on this server.")

for run in sorted(runs):
    zip_path = root / f"{run.name}_results.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for file in run.iterdir():
            if file.is_file() and file.suffix in {".json", ".csv", ".pt"}:
                archive.write(file, f"{run.name}/{file.name}")
    print("ZIP READY:", zip_path)


No completed training run found on this server.


In [12]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 8. Train the five-epoch pilot

An epoch is one pass through the 256 selected training clips. After each epoch,
evaluate the fixed validation rows without gradients. Select the checkpoint by
highest validation average precision; ties keep the earliest epoch. No threshold
tuning, weighting, resampling, or augmentation is performed. Each epoch saves
history, validation predictions, the current checkpoint, and the best checkpoint.
If all predictions are non-RD, report that outcome: accuracy alone is misleading
with this imbalance. A successful diagnostic does not guarantee useful validation
performance. This notebook does not resume partial training runs.


In [13]:
def run_epoch(model, loader, optimizer=None):
    fitting = optimizer is not None
    model.train(fitting)
    total_loss = 0.0
    labels, probabilities = [], []
    with torch.set_grad_enabled(fitting):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if fitting:
                optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            if not torch.isfinite(loss):
                raise ValueError('Nonfinite loss; training stopped')
            if fitting:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(y)
            labels.extend(y.detach().cpu().tolist())
            probabilities.extend(torch.sigmoid(logits.detach()).cpu().tolist())
    return total_loss / len(loader.dataset), np.asarray(labels), np.asarray(probabilities)

def save_checkpoint(name, payload):
    temporary = output_dir / (name + '.tmp')
    torch.save(payload, temporary)
    temporary.replace(output_dir / name)


In [14]:
if not diagnostics_passed:
    raise RuntimeError('Diagnostics must pass before fitting.')
seed_everything(config['seed'])
train_loader, val_loader = make_loaders()
model = make_model()
optimizer = torch.optim.Adam(model.parameters(), lr=config['training']['learning_rate'])
history = []
best_score, best_epoch = -float('inf'), None
training_start = time.perf_counter()
reference = classification_metrics(validation['label'], np.zeros(len(validation)), config['evaluation']['threshold'])
save_json('constant_non_rd_reference.json', reference)
print('Constant non-RD validation reference:', reference)
save_json('run_status.json', {'status': 'training', 'test_used': False})

for epoch in range(1, config['training']['epochs'] + 1):
    epoch_start = time.perf_counter()
    train_loss, _, _ = run_epoch(model, train_loader, optimizer)
    val_loss, val_labels, val_probabilities = run_epoch(model, val_loader)
    np.testing.assert_array_equal(val_labels, validation['label'].to_numpy())
    metrics = classification_metrics(val_labels, val_probabilities, config['evaluation']['threshold'])
    score = metrics['average_precision']
    if score is None or not np.isfinite(score):
        raise ValueError('Undefined checkpoint-selection metric')
    row = {'epoch': epoch, 'train_loss': train_loss, 'validation_loss': val_loss,
           'seconds': time.perf_counter() - epoch_start, **metrics}
    history.append(row)
    predictions = validation[['clip_id', 'file_path', 'label']].copy().rename(columns={'label': 'dataset_label'})
    predictions['rd_probability'] = val_probabilities
    predictions['predicted_label'] = (val_probabilities >= config['evaluation']['threshold']).astype(int)
    predictions.to_csv(output_dir / f'validation_epoch_{epoch:02d}.csv', index=False)
    checkpoint = {'epoch': epoch, 'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(), 'validation_average_precision': score,
        'config': config, 'config_sha256': config_hash, 'manifest_hashes': manifest_hashes,
        'runtime': runtime, 'test_used': False}
    save_checkpoint('last.pt', checkpoint)
    if improves_average_precision(score, best_score):
        best_score, best_epoch = score, epoch
        save_checkpoint('best.pt', checkpoint)
        predictions.to_csv(output_dir / 'best_validation_predictions.csv', index=False)
        save_json('best_validation_metrics.json', {'epoch': epoch, 'validation_loss': val_loss, **metrics})
    save_json('history.json', history)
    pd.DataFrame([{k: v for k, v in item.items() if k != 'undefined_reasons'}
                  for item in history]).to_csv(output_dir / 'history.csv', index=False)
    save_json('run_status.json', {'status': 'training', 'completed_epochs': epoch,
        'best_epoch': best_epoch, 'best_average_precision': best_score, 'test_used': False})
    print({key: row[key] for key in ['epoch', 'train_loss', 'validation_loss', 'average_precision',
        'sensitivity', 'specificity', 'precision', 'accuracy', 'seconds']}, flush=True)

training_seconds = time.perf_counter() - training_start
best_metrics = json.loads((output_dir / 'best_validation_metrics.json').read_text())
save_json('summary.json', {'experiment_id': config['experiment_id'], 'run_id': run_id,
    'completed_at_utc': datetime.now(timezone.utc).isoformat(), 'config_sha256': config_hash,
    'manifest_hashes': manifest_hashes, 'runtime': runtime, 'parameter_count': parameter_count,
    'diagnostics_passed': diagnostics_passed, 'training_clips': len(pilot), 'validation_clips': len(validation),
    'best_epoch': best_epoch, 'best_validation_metrics': best_metrics,
    'constant_non_rd_reference': reference, 'training_seconds': training_seconds,
    'preprocessing_seconds': preprocessing_seconds, 'test_used': False,
    'limitations': config['limitations']})
save_json('run_status.json', {'status': 'completed', 'completed_epochs': len(history),
    'best_epoch': best_epoch, 'test_used': False})
display(pd.DataFrame([best_metrics, reference], index=['selected model: validation', 'always non-RD: validation']))
print('Completed. Best epoch:', best_epoch, '| validation AP:', best_score)
print('Results:', output_dir)


Constant non-RD validation reference: {'TP': 0, 'TN': 391, 'FP': 0, 'FN': 40, 'sensitivity': 0.0, 'specificity': 1.0, 'precision': None, 'F1': 0.0, 'accuracy': 0.9071925754060325, 'threshold': 0.5, 'positive_class': 'rd', 'balanced_accuracy': 0.5, 'average_precision': 0.09280742459396751, 'ROC_AUC': 0.5, 'undefined_reasons': {'precision': 'zero denominator'}}
{'epoch': 1, 'train_loss': 0.4433215819299221, 'validation_loss': 0.35435398441594607, 'average_precision': 0.06554964583293213, 'sensitivity': 0.0, 'specificity': 1.0, 'precision': None, 'accuracy': 0.9071925754060325, 'seconds': 1.4354025739999088}
{'epoch': 2, 'train_loss': 0.33622301975265145, 'validation_loss': 0.32215442852465015, 'average_precision': 0.06587143021837044, 'sensitivity': 0.0, 'specificity': 1.0, 'precision': None, 'accuracy': 0.9071925754060325, 'seconds': 1.2636979379999502}
{'epoch': 3, 'train_loss': 0.3229308626614511, 'validation_loss': 0.3156905498554424, 'average_precision': 0.06610597222981043, 'sensit

,epoch,validation_loss,TP,TN,FP,FN,sensitivity,specificity,precision,F1,accuracy,threshold,positive_class,balanced_accuracy,average_precision,ROC_AUC,undefined_reasons
selected model: validation,5.0,0.311541,0,391,0,40,0.0,1.0,None,0.0,0.907193,0.5,rd,0.5,0.090683,0.449105,{'precision': 'zero denominator'}
always non-RD: validation,NaN,NaN,0,391,0,40,0.0,1.0,None,0.0,0.907193,0.5,rd,0.5,0.092807,0.500000,{'precision': 'zero denominator'}


Completed. Best epoch: 5 | validation AP: 0.09068340359421731
Results: /content/outputs/baseline_training/20261028T193629481701Z


## 9. Preserve the evidence before disconnecting

The ZIP includes configuration, environment, manifests, diagnostics, history,
validation predictions, and checkpoints. It excludes videos and preprocessed
arrays. Run this cell to create the ZIP, then download it through Colab's Files
pane or the link. If using Drive, the run folder is already persistent. Keep the
ZIP/checkpoints in ignored outputs, not Git. Download the executed notebook too
for the exact code used. Share `summary.json`, `history.csv`, and
`diagnostics.json` for the next review. Report failures from their saved evidence.

These are pilot validation findings. Do not evaluate the test set yet or describe
the result as clinical validation. The next research decision follows inspection
of this run, including false negatives and comparison with the constant reference.


In [15]:
from IPython.display import FileLink
export_path = output_dir.parent / f'{output_dir.name}_results.zip'
artifact_hashes = {path.name: sha256(path.read_bytes()) for path in sorted(output_dir.iterdir())
    if path.is_file() and path.suffix in {'.json', '.csv', '.pt'} and path.name != 'artifact_sha256.json'}
save_json('artifact_sha256.json', artifact_hashes)
with zipfile.ZipFile(export_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(output_dir.iterdir()):
        if path.is_file() and path.suffix in {'.json', '.csv', '.pt'}:
            archive.write(path, arcname=f'{output_dir.name}/{path.name}')
print('Saved results ZIP:', export_path)
display(FileLink(str(export_path)))
# In browser Colab, you may also run:
# from google.colab import files
# files.download(str(export_path))


Saved results ZIP: /content/outputs/baseline_training/20261028T193629481701Z_results.zip


/content/outputs/baseline_training/20261028T193629481701Z_results.zip

In [18]:
from pathlib import Path
import shutil
import hashlib

source = Path(
    "/content/outputs/baseline_training/"
    "20261028T193629481701Z_results.zip"
)
drive_root = Path("/content/drive/MyDrive")

if not source.is_file():
    raise FileNotFoundError(f"ZIP not found on this server: {source}")
if not drive_root.is_dir():
    raise RuntimeError("Google Drive is not mounted in this session.")

destination = (
    drive_root / "rd-ultrasound-ml" / "outputs"
    / "baseline_training" / source.name
)
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source, destination)

assert (
    hashlib.sha256(source.read_bytes()).digest()
    == hashlib.sha256(destination.read_bytes()).digest()
)
print("COPY VERIFIED — Google Drive:", destination)


COPY VERIFIED — Google Drive: /content/drive/MyDrive/rd-ultrasound-ml/outputs/baseline_training/20261028T193629481701Z_results.zip


## 10. Audit the completed baseline predictions

**Continue here in the same connected T4 session after section 9. Do not Run All
or rerun sections 1–9. Download the original results ZIP first.** These follow-up
cells preserve the completed baseline and reuse its arrays without downloading
or decoding videos again. If the runtime was reset, stop: the session variables
and prepared arrays need restoring; these cells are not a standalone notebook.

First independently recompute the saved validation metrics, verify row/label
alignment, and inspect the score distribution by dataset class. The initial
model predicted every clip as non-RD at threshold 0.5. That alone does not prove
class imbalance caused the outcome. A mismatch in saved metrics or labels stops
the follow-up before fitting. This section changes no threshold or model.


In [19]:
def audit_saved_predictions(predictions, manifest, reported, threshold):
    if len(predictions) != len(manifest) or not predictions['clip_id'].is_unique:
        raise ValueError('Saved predictions have missing or duplicate rows')
    for column in ['clip_id', 'file_path']:
        if predictions[column].tolist() != manifest[column].tolist():
            raise ValueError(f'Saved prediction/manifest {column} order differs')
    np.testing.assert_array_equal(predictions['dataset_label'], manifest['label'])
    probabilities = predictions['rd_probability'].to_numpy()
    metrics = classification_metrics(predictions['dataset_label'], probabilities, threshold)
    np.testing.assert_array_equal(predictions['predicted_label'], (probabilities >= threshold).astype(int))
    for key, actual in metrics.items():
        expected = reported[key]
        if isinstance(actual, (int, float)) and not isinstance(actual, bool):
            if expected is None or not np.isclose(actual, expected, rtol=1e-9, atol=1e-12):
                raise ValueError(f'Recomputed metric differs: {key}')
        elif actual != expected:
            raise ValueError(f'Recomputed metric differs: {key}')
    return metrics

def training_positive_weight(training_labels):
    labels = np.asarray(training_labels)
    if labels.ndim != 1 or not np.isin(labels, [0, 1]).all():
        raise ValueError('Expected binary training labels')
    positives = int(np.sum(labels == 1))
    negatives = int(np.sum(labels == 0))
    if positives == 0 or negatives == 0:
        raise ValueError('Both training classes are required for this comparison')
    return negatives / positives

def make_weighted_config(baseline_config, training_labels):
    value = json.loads(json.dumps(baseline_config))
    value['experiment_id'] = 'baseline_002_weighted_pilot'
    value['status'] = 'planned_not_run'
    value['parent_experiment_id'] = baseline_config['experiment_id']
    value['training']['loss'] = 'BCEWithLogitsLoss with training-only pos_weight'
    value['training']['pos_weight'] = training_positive_weight(training_labels)
    value['comparison'] = {
        'changed_variable': 'positive-class weight in training loss only',
        'weight_rule': 'number of non-RD training clips / number of RD training clips',
        'validation_loss': 'unweighted BCEWithLogitsLoss, same as baseline_001',
        'initialization': 'fresh model and Adam state with the same seed; no checkpoint fine-tuning',
        'baseline_selection': 'baseline_001 checkpoint selected by validation average precision',
        'threshold_tuning': 'none; fixed at 0.5 for this comparison',
    }
    return value

def write_followup_json(directory, name, value):
    path = directory / name
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + '\n', encoding='utf-8')
    temporary.replace(path)

def run_comparison_epoch(model, loader, loss_function, optimizer=None):
    fitting = optimizer is not None
    model.train(fitting)
    total_loss = 0.0
    labels, probabilities = [], []
    with torch.set_grad_enabled(fitting):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if fitting:
                optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = loss_function(logits, y)
            if not torch.isfinite(loss):
                raise ValueError('Nonfinite comparison loss')
            if fitting:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(y)
            labels.extend(y.detach().cpu().tolist())
            probabilities.extend(torch.sigmoid(logits.detach()).cpu().tolist())
    return total_loss / len(loader.dataset), np.asarray(labels), np.asarray(probabilities)

def write_followup_checkpoint(directory, name, value):
    temporary = directory / (name + '.tmp')
    torch.save(value, temporary)
    temporary.replace(directory / name)


In [17]:
baseline_audit_passed = False
if 'output_dir' not in globals() or 'train_array' not in globals() or 'val_array' not in globals():
    raise RuntimeError('Continue in the completed baseline session; do not restart the runtime.')
baseline_dir = output_dir
baseline_summary = json.loads((baseline_dir / 'summary.json').read_text())
baseline_status = json.loads((baseline_dir / 'run_status.json').read_text())
saved_baseline_config = json.loads((baseline_dir / 'config.json').read_text())
if baseline_status['status'] != 'completed' or saved_baseline_config != config:
    raise ValueError('A completed baseline with the unchanged configuration is required')
if baseline_summary['test_used'] or baseline_summary['experiment_id'] != 'baseline_001_pilot':
    raise ValueError('Unexpected parent run or test use')
if baseline_summary['config_sha256'] != sha256((baseline_dir / 'config.json').read_bytes()):
    raise ValueError('Baseline configuration checksum differs')
for name, expected_hash in baseline_summary['manifest_hashes'].items():
    if sha256((baseline_dir / name).read_bytes()) != expected_hash:
        raise ValueError(f'Baseline manifest was changed: {name}')
for name, frame in [('train_manifest.csv', pilot), ('validation_manifest.csv', validation)]:
    if frame.to_csv(index=False, lineterminator='\n').encode() != (baseline_dir / name).read_bytes():
        raise ValueError('Current manifest/order differs from the completed baseline')
if len(train_array) != len(pilot) or len(val_array) != len(validation):
    raise ValueError('Prepared array/manifest lengths differ')
baseline_predictions = pd.read_csv(baseline_dir / 'best_validation_predictions.csv')
audited_metrics = audit_saved_predictions(baseline_predictions, validation,
    baseline_summary['best_validation_metrics'], config['evaluation']['threshold'])
review_dir = baseline_dir.parent / (baseline_dir.name + '_review')
review_dir.mkdir(parents=True, exist_ok=True)
score_summary = baseline_predictions.groupby('dataset_label')['rd_probability'].describe(percentiles=[.1, .5, .9])
score_summary.to_csv(review_dir / 'baseline_probability_summary.csv')
write_followup_json(review_dir, 'baseline_prediction_audit.json', {
    'parent_run': baseline_dir.name, 'metrics_recomputed': audited_metrics,
    'prediction_csv_sha256': sha256((baseline_dir / 'best_validation_predictions.csv').read_bytes()),
    'manifest_hashes': baseline_summary['manifest_hashes'], 'test_used': False,
})
display(score_summary.rename(index={0: 'Dataset non-RD', 1: 'Dataset RD'}))
print('PASS: saved validation predictions, labels, metrics, and manifest order agree.')
print('Predicted RD clips at threshold 0.5:', int(baseline_predictions['predicted_label'].sum()))
print('Review artifacts:', review_dir)
baseline_audit_passed = True


,count,mean,std,min,10%,50%,90%,max
dataset_label,,,,,,,,
Dataset non-RD,391.0,0.075287,0.003276,0.055038,0.071509,0.075908,0.078390,0.081777
Dataset RD,40.0,0.074952,0.002971,0.068078,0.070742,0.075697,0.077616,0.080258


PASS: saved validation predictions, labels, metrics, and manifest order agree.
Predicted RD clips at threshold 0.5: 0
Review artifacts: /content/outputs/baseline_training/20261028T193629481701Z_review


## 11. Controlled comparison: weight RD examples in the training loss

Hypothesis: giving the minority training class more loss weight may reduce the
all-non-RD outcome. This is a hypothesis, not a guaranteed improvement.
Use `pos_weight = training non-RD / training RD = 232/24 ≈ 9.67`, computed only
from the unchanged training manifest. Keep the same seed, initial model, loader
ordering, 256 training/431 validation clips, five epochs, Adam 0.001, batch size 8,
input arrays, checkpoint rule, and threshold 0.5. No resampling or threshold tuning.
[PyTorch pos_weight documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html).

Run both code cells below after the section 10 audit passes. The first creates
a new run folder and records its configuration/manifests. The second initializes
a **fresh** model and optimizer; it does not fine-tune the previous checkpoint.
Only the training loss is weighted. Validation loss remains unweighted so it can
be compared with baseline_001. Weighted training loss is on a different scale
and must not be compared directly with the previous unweighted training loss.

Weighting can change the meaning of sigmoid scores; do not assume that these
scores are calibrated clinical probabilities. Judge AP, ROC-AUC, sensitivity,
and specificity together. An all-RD result also fails to establish useful
discrimination. A lower accuracy alone is not evidence of failure or success.


In [20]:
weighted_proposal = {'experiment_id': 'baseline_002_weighted_pilot',
 'status': 'planned_not_run',
 'task': 'non_rd_vs_rd',
 'label_mapping': {'non_rd': 0, 'rd': 1},
 'seed': 42,
 'split_policy': 'preserve official train/val/test membership; no test use during '
                 'development',
 'preprocessing': {'num_frames': 16,
                   'height': 128,
                   'width': 128,
                   'frame_sampling': 'uniform frame indices using rint(linspace(0, '
                                     'N-1, T)); repeats for short clips',
                   'grayscale': 'arithmetic channel mean',
                   'padding': 'symmetric zero padding to square',
                   'resize': 'OpenCV INTER_AREA',
                   'normalization': 'divide by 255',
                   'layout': 'B,C,T,H,W',
                   'dtype': 'float32',
                   'augmentation': 'none'},
 'model': {'name': 'compact_3d_cnn_from_scratch',
           'channels': [8, 16, 32],
           'head': 'global average pooling and one logit',
           'pretrained': False,
           'blocks': 'Conv3d(kernel=3,stride=2,padding=1,bias=True), ReLU; repeated '
                     'for each channel count',
           'pool': 'global mean over temporal and spatial dimensions',
           'output': 'Linear(32,1), no sigmoid'},
 'training': {'device': 'Colab CUDA GPU',
              'optimizer': 'Adam',
              'learning_rate': 0.001,
              'batch_size': 8,
              'epochs': 5,
              'loss': 'BCEWithLogitsLoss with training-only pos_weight',
              'pilot_training_target': 256,
              'validation': 'all 431 supplied validation clips',
              'pilot_selection': 'deterministic proportional class sample within '
                                 'official training only; record manifest before '
                                 'fitting',
              'pilot_class_counts': {'0': 232, '1': 24},
              'num_workers': 0,
              'shuffle': True,
              'precision': 'float32',
              'deterministic_algorithms': True,
              'pos_weight': 9.666666666666666},
 'evaluation': {'positive_class': 'rd',
                'threshold': 0.5,
                'checkpoint_selection': 'highest validation average precision; '
                                        'earliest epoch on ties',
                'metrics': ['TP',
                            'TN',
                            'FP',
                            'FN',
                            'sensitivity',
                            'specificity',
                            'precision',
                            'F1',
                            'balanced_accuracy',
                            'average_precision',
                            'ROC_AUC',
                            'accuracy'],
                'final_test': 'reserved; no test loading, inference, tuning or '
                              'checkpoint selection'},
 'limitations': ['clip-level splits allow same-patient overlap; no unseen-patient '
                 'generalization claim',
                 'eight inspection clips are not a training dataset or a metric '
                 'estimate',
                 'pilot is not a reproduction of the paper or a final experiment'],
 'preparation_config_sha256': '40753e7e9a4ba8bca86b445307a26c85e2087ac3740b93cf1525ae4d2fe7a1e1',
 'diagnostics': {'clips_per_class': 2,
                 'max_steps': 300,
                 'learning_rate': 0.003,
                 'loss_ratio_required': 0.5,
                 'training_accuracy_required': 1.0,
                 'selection': 'first two per class in the ranked pilot manifest; '
                              'training only'},
 'parent_experiment_id': 'baseline_001_pilot',
 'comparison': {'changed_variable': 'positive-class weight in training loss only',
                'weight_rule': 'number of non-RD training clips / number of RD '
                               'training clips',
                'validation_loss': 'unweighted BCEWithLogitsLoss, same as baseline_001',
                'initialization': 'fresh model and Adam state with the same seed; no '
                                  'checkpoint fine-tuning',
                'baseline_selection': 'baseline_001 checkpoint selected by validation '
                                      'average precision',
                'threshold_tuning': 'none; fixed at 0.5 for this comparison'}}

if not baseline_audit_passed:
    raise RuntimeError('Complete the section 10 audit first.')
if not torch.cuda.is_available():
    raise RuntimeError('The weighted comparison requires the connected CUDA runtime.')
weighted_config = make_weighted_config(config, pilot['label'].to_numpy())
if weighted_config != weighted_proposal:
    raise ValueError('Current configuration differs from configs/baseline_002_weighted.json')
weighted_run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '_weighted'
weighted_dir = baseline_dir.parent / weighted_run_id
weighted_dir.mkdir(parents=True, exist_ok=False)
write_followup_json(weighted_dir, 'config.json', weighted_config)
weighted_config_hash = sha256((weighted_dir / 'config.json').read_bytes())
for name in ['train_manifest.csv', 'validation_manifest.csv', 'diagnostic_manifest.csv',
             'official_train.csv', 'official_val.csv', 'manifests.json', 'runtime.json',
             'preprocessing_implementation.json', 'model.json']:
    (weighted_dir / name).write_bytes((baseline_dir / name).read_bytes())
write_followup_json(weighted_dir, 'parent_run.json', {
    'run_id': baseline_dir.name, 'experiment_id': config['experiment_id'],
    'config_sha256': baseline_summary['config_sha256'],
    'summary_sha256': sha256((baseline_dir / 'summary.json').read_bytes()),
    'manifest_hashes': baseline_summary['manifest_hashes'],
    'baseline_metrics': audited_metrics, 'test_used': False,
})
write_followup_json(weighted_dir, 'run_status.json', {'status': 'configured', 'test_used': False})
print('Training-only pos_weight:', weighted_config['training']['pos_weight'])
print('New results:', weighted_dir)
print('Prepared arrays reused. Original baseline artifacts remain in:', baseline_dir)


Training-only pos_weight: 9.666666666666666
New results: /content/outputs/baseline_training/20261028T201752658036Z_weighted
Prepared arrays reused. Original baseline artifacts remain in: /content/outputs/baseline_training/20261028T193629481701Z


In [21]:
if not baseline_audit_passed:
    raise RuntimeError('The baseline audit must pass before the comparison.')
if (weighted_dir / 'history.json').exists() or (weighted_dir / 'last.pt').exists():
    raise RuntimeError('This comparison has training artifacts. Rerun its configuration cell to create a fresh run.')
seed_everything(weighted_config['seed'])
weighted_train_loader, weighted_val_loader = make_loaders()
weighted_model = make_model()
weighted_optimizer = torch.optim.Adam(weighted_model.parameters(), lr=weighted_config['training']['learning_rate'])
positive_weight = torch.tensor([weighted_config['training']['pos_weight']], dtype=torch.float32, device=device)
weighted_loss = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
unweighted_validation_loss = nn.BCEWithLogitsLoss()
weighted_history = []
weighted_best_score, weighted_best_epoch = -float('inf'), None
weighted_start = time.perf_counter()
write_followup_json(weighted_dir, 'run_status.json', {'status': 'training', 'test_used': False})
for epoch in range(1, weighted_config['training']['epochs'] + 1):
    epoch_start = time.perf_counter()
    training_loss, _, _ = run_comparison_epoch(weighted_model, weighted_train_loader, weighted_loss, weighted_optimizer)
    validation_loss, labels, probabilities = run_comparison_epoch(weighted_model, weighted_val_loader, unweighted_validation_loss)
    np.testing.assert_array_equal(labels, validation['label'].to_numpy())
    metrics = classification_metrics(labels, probabilities, weighted_config['evaluation']['threshold'])
    score = metrics['average_precision']
    if score is None or not np.isfinite(score):
        raise ValueError('Undefined comparison checkpoint-selection metric')
    row = {'epoch': epoch, 'weighted_train_loss': training_loss, 'validation_loss': validation_loss,
        'seconds': time.perf_counter() - epoch_start, **metrics}
    weighted_history.append(row)
    predictions = validation[['clip_id', 'file_path', 'label']].copy().rename(columns={'label': 'dataset_label'})
    predictions['rd_probability'] = probabilities
    predictions['predicted_label'] = (probabilities >= weighted_config['evaluation']['threshold']).astype(int)
    predictions.to_csv(weighted_dir / f'validation_epoch_{epoch:02d}.csv', index=False)
    checkpoint = {'epoch': epoch, 'model_state_dict': weighted_model.state_dict(),
        'optimizer_state_dict': weighted_optimizer.state_dict(), 'validation_average_precision': score,
        'config': weighted_config, 'config_sha256': weighted_config_hash,
        'manifest_hashes': baseline_summary['manifest_hashes'], 'runtime': runtime, 'test_used': False}
    write_followup_checkpoint(weighted_dir, 'last.pt', checkpoint)
    if improves_average_precision(score, weighted_best_score):
        weighted_best_score, weighted_best_epoch = score, epoch
        write_followup_checkpoint(weighted_dir, 'best.pt', checkpoint)
        predictions.to_csv(weighted_dir / 'best_validation_predictions.csv', index=False)
        write_followup_json(weighted_dir, 'best_validation_metrics.json', {'epoch': epoch, 'validation_loss': validation_loss, **metrics})
    write_followup_json(weighted_dir, 'history.json', weighted_history)
    pd.DataFrame([{k: v for k, v in item.items() if k != 'undefined_reasons'}
                  for item in weighted_history]).to_csv(weighted_dir / 'history.csv', index=False)
    write_followup_json(weighted_dir, 'run_status.json', {'status': 'training', 'completed_epochs': epoch,
        'best_epoch': weighted_best_epoch, 'test_used': False})
    print({key: row[key] for key in ['epoch', 'weighted_train_loss', 'validation_loss', 'average_precision',
        'sensitivity', 'specificity', 'precision', 'accuracy', 'seconds']}, flush=True)

weighted_best_metrics = json.loads((weighted_dir / 'best_validation_metrics.json').read_text())
comparison_table = pd.DataFrame([
    baseline_summary['best_validation_metrics'], weighted_best_metrics,
    baseline_summary['constant_non_rd_reference']],
    index=['baseline_001: validation', 'baseline_002 weighted: validation', 'always non-RD: validation'])
comparison_table.to_csv(weighted_dir / 'comparison.csv', index_label='model')
write_followup_json(weighted_dir, 'summary.json', {
    'experiment_id': weighted_config['experiment_id'], 'run_id': weighted_dir.name,
    'completed_at_utc': datetime.now(timezone.utc).isoformat(), 'config_sha256': weighted_config_hash,
    'parent_run_id': baseline_dir.name, 'manifest_hashes': baseline_summary['manifest_hashes'],
    'runtime': runtime, 'training_clips': len(pilot), 'validation_clips': len(validation),
    'parameter_count': sum(p.numel() for p in weighted_model.parameters()),
    'training_only_pos_weight': weighted_config['training']['pos_weight'],
    'best_epoch': weighted_best_epoch, 'best_validation_metrics': weighted_best_metrics,
    'baseline_best_validation_metrics': baseline_summary['best_validation_metrics'],
    'constant_non_rd_reference': baseline_summary['constant_non_rd_reference'],
    'training_seconds': time.perf_counter() - weighted_start, 'test_used': False,
    'limitations': weighted_config['limitations'] + [
        'single-seed small pilot; weighting is a controlled comparison, not proof of the cause',
        'weighted sigmoid scores are not established calibrated probabilities'],
})
write_followup_json(weighted_dir, 'run_status.json', {'status': 'completed', 'completed_epochs': len(weighted_history),
    'best_epoch': weighted_best_epoch, 'test_used': False})
display(comparison_table)
print('Weighted comparison completed. Best epoch:', weighted_best_epoch)
print('Validation AP change vs baseline:', weighted_best_score - audited_metrics['average_precision'])
print('Results:', weighted_dir)


{'epoch': 1, 'weighted_train_loss': 1.2728406637907028, 'validation_loss': 0.6201967897105383, 'average_precision': 0.13596161533174309, 'sensitivity': 0.0, 'specificity': 1.0, 'precision': None, 'accuracy': 0.9071925754060325, 'seconds': 1.4407782759999463}
{'epoch': 2, 'weighted_train_loss': 1.264043491333723, 'validation_loss': 0.6264823590796398, 'average_precision': 0.13293321693103333, 'sensitivity': 0.0, 'specificity': 1.0, 'precision': None, 'accuracy': 0.9071925754060325, 'seconds': 1.3081130760001543}
{'epoch': 3, 'weighted_train_loss': 1.262833695858717, 'validation_loss': 0.634100260286486, 'average_precision': 0.1358513031699346, 'sensitivity': 0.0, 'specificity': 1.0, 'precision': None, 'accuracy': 0.9071925754060325, 'seconds': 1.31687005699996}
{'epoch': 4, 'weighted_train_loss': 1.2618171703070402, 'validation_loss': 0.6376111501884018, 'average_precision': 0.12753864177753915, 'sensitivity': 0.0, 'specificity': 1.0, 'precision': None, 'accuracy': 0.9071925754060325, '

,epoch,validation_loss,TP,TN,FP,FN,sensitivity,specificity,precision,F1,accuracy,threshold,positive_class,balanced_accuracy,average_precision,ROC_AUC,undefined_reasons
baseline_001: validation,5.0,0.311541,0,391,0,40,0.0,1.0,None,0.0,0.907193,0.5,rd,0.5,0.090683,0.449105,{'precision': 'zero denominator'}
baseline_002 weighted: validation,5.0,0.642468,0,391,0,40,0.0,1.0,None,0.0,0.907193,0.5,rd,0.5,0.145320,0.611061,{'precision': 'zero denominator'}
always non-RD: validation,NaN,NaN,0,391,0,40,0.0,1.0,None,0.0,0.907193,0.5,rd,0.5,0.092807,0.500000,{'precision': 'zero denominator'}


Weighted comparison completed. Best epoch: 5
Validation AP change vs baseline: 0.054636574224222606
Results: /content/outputs/baseline_training/20261028T201752658036Z_weighted


## 12. Export the weighted run and compare the results

Download this new ZIP as well as the original baseline ZIP. Share the comparison
table, `summary.json`, and `baseline_probability_summary.csv` for review. Preserve
the executed notebook. If weighting fails to improve ranking or produces all-RD
predictions, record that outcome before changing another variable. Keep the test
set reserved. Do not tune a threshold as part of this experiment.


In [22]:
weighted_artifact_hashes = {path.name: sha256(path.read_bytes()) for path in sorted(weighted_dir.iterdir())
    if path.is_file() and path.suffix in {'.json', '.csv', '.pt'} and path.name != 'artifact_sha256.json'}
write_followup_json(weighted_dir, 'artifact_sha256.json', weighted_artifact_hashes)
weighted_export_path = weighted_dir.parent / (weighted_dir.name + '_results.zip')
with zipfile.ZipFile(weighted_export_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for directory in [weighted_dir, review_dir]:
        for path in sorted(directory.iterdir()):
            if path.is_file() and path.suffix in {'.json', '.csv', '.pt'}:
                archive.write(path, arcname=f'{directory.name}/{path.name}')
print('Saved weighted results ZIP:', weighted_export_path)
display(FileLink(str(weighted_export_path)))
# In browser Colab, you may also run:
# from google.colab import files
# files.download(str(weighted_export_path))


Saved weighted results ZIP: /content/outputs/baseline_training/20261028T201752658036Z_weighted_results.zip


/content/outputs/baseline_training/20261028T201752658036Z_weighted_results.zip